# 06 — Ponte verso la fase agente / RL
Un decision model è un **router/guard** rapido ed economico dentro un loop agente: sceglie la prossima azione con una probabilità, mentre codice deterministico applica le regole e le decisioni incerte passano a un modello più forte (o a una persona).

La Fase 2 inquadra un piccolo task sequenziale come obiettivo in stile RL: `ricompensa = successo - costo_per_passo`. Lo simuliamo con il mock gratuito.


In [ ]:
import random
from decision_qa import MockClient, ChoiceQuestion

client = MockClient(seed=7)
ACTIONS = ['retrieve', 'answer', 'escalate']

def agent_step(state, rng):
    # Simulate a policy that has learned to retrieve before answering.
    target = 0 if not state['has_evidence'] else 1
    q = ChoiceQuestion(label='next_action', options=ACTIONS, gold_index=target)
    result = client.decision(state, [q])[0]
    action = max(result.probabilities, key=result.probabilities.get)
    confidence = result.probabilities[action]
    # deterministic policy: never allow 'answer' without enough evidence
    if action == 'answer' and not state['has_evidence']:
        action = 'retrieve'
    if confidence < 0.6:
        action = 'escalate'
    return action, confidence


In [ ]:
def episode(rng, max_steps=4):
    state = {'question': 'What did the trial report?', 'has_evidence': False}
    steps = 0
    while steps < max_steps:
        action, conf = agent_step(state, rng)
        steps += 1
        if action == 'retrieve':
            state['has_evidence'] = True
        elif action == 'answer':
            return 1.0 - 0.1 * steps  # success minus step cost
        else:
            return -0.2  # escalated: small penalty, no wrong answer
    return -0.1

rng = random.Random(0)
rewards = [episode(rng) for _ in range(20)]
print('mean reward:', round(sum(rewards)/len(rewards), 3))


## Prossimi passi
- Sostituisci il mock con un modello open locale tramite `SystemOneClient`.
- Aggiungi un task e una reward reali (es. QA multi-hop: reward = exact match - costo latenza).
- Studia loop agente open come [JevLoop](https://github.com/zjunlp/JevLoop) e [ProgressGate](https://github.com/AshutoshVJTI/progressgate).

Tutti i link e gli strumenti usati in questo repository sono liberi/open.
